In [11]:
import yaml
from pathlib import Path
import re
from dataclasses import dataclass
from typing import List, Any
import ast

In [2]:
@dataclass(frozen=True)
class HPRow:
    model: str
    hp_name: str
    domain: str
    is_log: bool = False

In [3]:
LATEX_SPECIALS = {
    "&": r"\&", "%": r"\%", "$": r"\$", "#": r"\#",
    "_": r"\_", "{": r"\{", "}": r"\}",
    "~": r"\textasciitilde{}", "^": r"\textasciicircum{}",
    # "\\": r"\textbackslash{}",
}

def latex_escape(s: str) -> str:
    return "".join(LATEX_SPECIALS.get(c, c) for c in s)

def model_name_from_path(path: Path) -> str:
    return re.sub(r"\s+", " ", path.stem.strip())

def as_str(v: Any) -> str:
    if v is None:
        return ""
    if isinstance(v, bool):
        return "true" if v else "false"
    return str(v)

def domain_from_hp(hp: dict) -> str:
    if "choices" in hp and isinstance(hp["choices"], list):
        return "{" + ", ".join(as_str(c) for c in hp["choices"]) + "}"

    lower = hp.get("lower")
    upper = hp.get("upper")
    if lower is not None or upper is not None:
        return f"[{as_str(lower)}, {as_str(upper)}]"
    return ""

def latex_escape_no_bs(s: str) -> str:
    # like latex_escape, but do NOT escape backslashes
    specials = {
        "&": r"\&", "%": r"\%", "$": r"\$", "#": r"\#",
        "_": r"\_", "{": r"\{", "}": r"\}",
        "~": r"\textasciitilde{}", "^": r"\textasciicircum{}",
        # "\\": r"\textbackslash{}",  # intentionally omitted
    }
    return "".join(specials.get(c, c) for c in s)

def latex_range(s: str) -> str:
    s = latex_escape_no_bs(s)                # 1) escape YAML text
    s = s.replace(", ", ", ")                # normalize
    s = s.replace(",", ",\\linebreak[1] ")   # 2) inject LaTeX (not escaped)
    return s



In [4]:
def as_str(v: Any) -> str:
    if v is None:
        return ""
    if isinstance(v, bool):
        return "true" if v else "false"
    return str(v)


def is_learning_rate(name: str) -> bool:
    name = name.lower()
    return name == "lr" or "learning_rate" in name


def parse_range(value: Any) -> tuple[Any, Any] | None:
    if not isinstance(value, str):
        return None

    value = value.strip()
    if not (value.startswith("(") and value.endswith(")")):
        return None

    parsed = ast.literal_eval(value)
    if isinstance(parsed, tuple) and len(parsed) == 2:
        return parsed

    return None


def domain_from_value(name: str, value: Any) -> str | None:
    if isinstance(value, list):
        return r"\{" + ", ".join(latex_escape(as_str(v)) for v in value) + r"\}"

    r = parse_range(value)
    if r is None:
        return None  # constant → ignored

    lo, hi = r
    is_int = isinstance(lo, int) and isinstance(hi, int)
    left, right = ("[", "]") if is_int else ("(", ")")

    dagger = r"$^\dagger$" if is_learning_rate(name) else ""
    return f"{left}{latex_escape(as_str(lo))}, {latex_escape(as_str(hi))}{right}{dagger}"


def parse_yaml_file(path: Path) -> list[HPRow]:
    data = yaml.safe_load(path.read_text())

    if not isinstance(data, dict):
        return []

    model = model_name_from_path(path)
    rows: list[HPRow] = []

    for name, value in data.items():
        name = as_str(name).strip()
        if not name:
            continue

        domain = domain_from_value(name, value)
        if domain is None:
            continue

        rows.append(
            HPRow(
                model=model,
                hp_name=name,
                domain=domain,
                is_log=is_learning_rate(name),
            )
        )

    return rows

In [5]:
def texttt(s: str) -> str:
    return rf"\texttt{{{latex_escape(s)}}}"


def format_params(params: list[str]) -> str:
    return ", ".join(texttt(p) for p in sorted(params, key=str.lower))


def to_latex_longtable(rows: list[HPRow]) -> str:
    rows = sorted(rows, key=lambda r: (r.model.lower(), r.domain, r.hp_name.lower()))

    grouped_by_model: dict[str, list[HPRow]] = {}
    for r in rows:
        grouped_by_model.setdefault(r.model, []).append(r)

    lines = []
    lines.append(r"\large")
    lines.append(r"\setlength{\tabcolsep}{4pt}")
    lines.append(r"\begin{longtable}{l p{7.2cm} p{4.6cm}}")
    lines.append(
        r"\caption{Hyperparameter search spaces. A dagger ($^\dagger$) indicates that the corresponding range is sampled on a logarithmic scale.}"
        r"\label{tab:search_spaces}\\"
    )
    lines.append(r"\toprule")
    lines.append(r"Model & Parameters & Range \\")
    lines.append(r"\midrule")
    lines.append(r"\endfirsthead")
    lines.append(r"\toprule")
    lines.append(r"Model & Parameters & Range \\")
    lines.append(r"\midrule")
    lines.append(r"\endhead")

    for model, model_rows in grouped_by_model.items():
        grouped_by_domain: dict[str, list[str]] = {}
        for r in model_rows:
            grouped_by_domain.setdefault(r.domain, []).append(r.hp_name)

        domain_items = sorted(grouped_by_domain.items(), key=lambda x: (x[0], x[1][0].lower()))

        for i, (domain, params) in enumerate(domain_items):
            model_cell = latex_escape(model) if i == 0 else ""
            lines.append(
                f"{model_cell} & {format_params(params)} & {domain} \\\\"
            )

        lines.append(r"\midrule")

    lines.append(r"\end{longtable}")
    return "\n".join(lines)

In [26]:
def parse_trainer_yaml_file(path: Path) -> list[tuple[str, str, bool]]:
    data = yaml.safe_load(path.read_text())

    if not isinstance(data, dict):
        return []

    rows: list[tuple[str, str, bool]] = []

    for name, value in data.items():
        name = as_str(name).strip()
        if not name:
            continue

        domain = domain_from_value(name, value)
        if domain is None:
            continue

        rows.append((name, domain, is_learning_rate(name)))

    return rows

def collect_search_space_rows(
    search_space_dir: Path,
    trainer_path: str = "trainer.yml",
) -> list[HPRow]:
    search_space_paths = sorted(search_space_dir.glob("*.yml"))

    trainer_hps = (
        parse_trainer_yaml_file(trainer_path)
        if trainer_path.exists()
        else []
    )

    model_paths = [
        path for path in search_space_paths
    ]

    all_rows: list[HPRow] = []

    for path in model_paths:
        model = model_name_from_path(path)

        # Model-specific HPs
        all_rows.extend(parse_yaml_file(path))

        # Shared trainer HPs added to every model
        all_rows.extend(
            HPRow(
                model=model,
                hp_name=name,
                domain=domain,
                is_log=is_log,
            )
            for name, domain, is_log in trainer_hps
        )

    return all_rows

from collections import defaultdict

def split_shared_vs_model_specific(all_rows: list[HPRow]):
    # group rows per model
    per_model: dict[str, set[tuple[str, str, bool]]] = defaultdict(set)

    for r in all_rows:
        per_model[r.model].add((r.hp_name, r.domain, r.is_log))

    models = list(per_model.keys())

    # shared = intersection across all models
    shared = set.intersection(*per_model.values()) if models else set()

    # build outputs
    shared_rows = [
        HPRow(model="ALL", hp_name=n, domain=d, is_log=l)
        for (n, d, l) in sorted(shared)
    ]

    model_specific_rows: list[HPRow] = []
    for model, rows in per_model.items():
        specific = rows - shared
        model_specific_rows.extend(
            HPRow(model=model, hp_name=n, domain=d, is_log=l)
            for (n, d, l) in sorted(specific)
        )

    return shared_rows, model_specific_rows

In [27]:
all_rows = collect_search_space_rows(Path("../configs/model_configs"), Path("../configs/trainer_search_space.yml"))
shared_rows, model_specific_rows = split_shared_vs_model_specific(all_rows)
tex = to_latex_longtable(shared_rows)
print(tex)
print("\n\n\n\n\n\n")
tex = to_latex_longtable(model_specific_rows)
print(tex)

\large
\setlength{\tabcolsep}{4pt}
\begin{longtable}{l p{7.2cm} p{4.6cm}}
\caption{Hyperparameter search spaces. A dagger ($^\dagger$) indicates that the corresponding range is sampled on a logarithmic scale.}\label{tab:search_spaces}\\
\toprule
Model & Parameters & Range \\
\midrule
\endfirsthead
\toprule
Model & Parameters & Range \\
\midrule
\endhead
ALL & \texttt{learning\_rate} & (1e-05, 0.01)$^\dagger$ \\
 & \texttt{max\_epoch} & [1, 10] \\
 & \texttt{num\_layers} & \{1, 2, 3\} \\
 & \texttt{batch\_size} & \{8, 16, 32, 64, 128\} \\
 & \texttt{hidden\_size}, \texttt{time\_emb\_size} & \{8, 16, 32, 64\} \\
 & \texttt{use\_tfb} & \{false, true\} \\
\midrule
\end{longtable}







\large
\setlength{\tabcolsep}{4pt}
\begin{longtable}{l p{7.2cm} p{4.6cm}}
\caption{Hyperparameter search spaces. A dagger ($^\dagger$) indicates that the corresponding range is sampled on a logarithmic scale.}\label{tab:search_spaces}\\
\toprule
Model & Parameters & Range \\
\midrule
\endfirsthead
\toprule
